# MIMO-OTFS系统模型、预编码与联合检测

**MIMO-OTFS: System Model, Precoding, and Joint Detection**

---

本notebook介绍MIMO-OTFS（多输入多输出正交时频空）系统的完整技术框架。

前置知识：
- OTFS调制原理（延迟-多普勒域、SFFT/ISFFT）
- MIMO基础（空间复用、预编码、均衡）
- Zak变换（延迟-多普勒域与时频域的数学变换）

## 1. 目标 (Objectives)

通过本notebook，您将：

- **理解MIMO-OTFS系统模型**：掌握延迟-多普勒域MIMO信道的数学表示
- **掌握发送端预处理**：Zak变换、SFFT在MIMO系统中的应用
- **掌握接收端后处理**：ISFFT、IZak变换实现
- **理解MIMO预编码技术**：ZF、MMSE、Tomlinson-Harashima预编码
- **理解联合检测算法**：ML、MAP、消息传递检测器
- **通过Python演示**：完整MIMO-OTFS系统仿真

## 2. MIMO-OTFS系统模型

### 2.1 为什么将OTFS与MIMO结合？

MIMO和OTFS各有优势：

| 技术 | 优势 | 局限性 |
|------|------|--------|
| **MIMO** | 空间复用增益、阵列增益、分集增益 | 时变信道处理复杂、高移动性下性能下降 |
| **OTFS** | 时不变等价信道、单抽头均衡、天然处理多普勒 | 单天线容量受限 |

**MIMO-OTFS**结合两者优势：
- MIMO的空间复用能力
- OTFS在延迟-多普勒域的时不变信道特性
- 在高移动性场景下仍保持MIMO容量优势

### 2.2 MIMO-OTFS信号模型

考虑$N_t$发射天线、$N_r$接收天线的MIMO-OTFS系统：

**发送端**：
在延迟-多普勒域，每根发射天线$n_t$的发送信号表示为$X_{n_t}(\tau, \nu)$，构成延迟-多普勒域信号矩阵：

$$\mathbf{X}(\tau, \nu) = \begin{bmatrix} X_1(\tau, \nu) \\ X_2(\tau, \nu) \\ \vdots \\ X_{N_t}(\tau, \nu) \end{bmatrix} \in \mathbb{C}^{N_t \times MN}$$

其中$M$是频率（延迟）维度采样数，$N$是时间（多普勒）维度采样数。

**信道**：
每对发射-接收天线之间的信道在延迟-多普勒域表示为：

$$h_{r,n_t}(\tau, \nu) = \sum_{p=0}^{P-1} h_{r,n_t,p} \delta(\tau - \tau_p) \delta(\nu - \nu_p)$$

其中$P$是多径数量，$\tau_p$和$\nu_p$是第$p$条路径的延迟和多普勒偏移。

**MIMO信道矩阵**：

$$\mathbf{H}(\tau, \nu) = \begin{bmatrix} h_{1,1}(\tau,\nu) & h_{1,2}(\tau,\nu) & \cdots & h_{1,N_t}(\tau,\nu) \\ h_{2,1}(\tau,\nu) & h_{2,2}(\tau,\nu) & \cdots & h_{2,N_t}(\tau,\nu) \\ \vdots & \vdots & \ddots & \vdots \\ h_{N_r,1}(\tau,\nu) & h_{N_r,2}(\tau,\nu) & \cdots & h_{N_r,N_t}(\tau,\nu) \end{bmatrix} \in \mathbb{C}^{N_r \times N_t}$$

**接收信号**：

$$\mathbf{Y}(\tau, \nu) = \mathbf{H}(\tau, \nu) \mathbf{X}(\tau, \nu) + \mathbf{N}(\tau, \nu)$$

其中$\mathbf{N}(\tau, \nu)$是加性噪声。

### 2.3 Zak变换与SFFT

Zak变换是连接延迟-多普勒域与时频域的关键工具：

**连续Zak变换**：

$$[\mathcal{Z}x](t, f) = \sum_{k=-\infty}^{\infty} x(t + kT) e^{-j2\pi f kT}$$

**离散Zak变换（SFFT）**：

对于$M \times N$网格：

$$X_{DD}[m,n] \xrightarrow{\text{SFFT}} X_{TF}[k,l] = \frac{1}{N} \sum_{n=0}^{N-1} \sum_{m=0}^{M-1} X_{DD}[m,n] e^{j2\pi (\frac{kn}{N} + \frac{lm}{M})}$$

**MIMO系统的SFFT**：

对每根发射天线独立应用SFFT：

$$\mathbf{X}_{TF}(\tau, \nu) = \text{SFFT}(\mathbf{X}_{DD}(\tau, \nu))$$

其中$\mathbf{X}_{DD}(\tau, \nu)$是$N_t \times M \times N$的三维张量。

## 3. 代码演示：MIMO-OTFS系统参数

首先定义系统参数和辅助函数。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("MIMO-OTFS系统演示包加载成功")

In [ ]:
# MIMO-OTFS系统参数
N_t = 2   # 发射天线数
N_r = 2   # 接收天线数
N = 32    # 时间维度（多普勒采样数）/ OFDM符号数
M = 32    # 频率维度（延迟采样数）/ 子载波数
cp_len = 8  # 循环前缀长度

# QAM调制参数
qam_order = 16  # 16-QAM
qam_bits = 4    # 每符号比特数

# 信道参数
num_paths = 3   # 多径数量
max_delay = 5   # 最大延迟（采样点）
max_doppler = 3  # 最大多普勒偏移

print(f"MIMO配置: {N_t}x{N_r}")
print(f"OTFS网格大小: {N} x {M}")
print(f"总符号数: {N_t * N * M}")
print(f"循环前缀长度: {cp_len}")

## 4. 发送端预处理

### 4.1 Zak变换（Zak Transform）

Zak变换将信号从时域映射到时频-域，是OTFS的核心数学工具。

对于MIMO系统，每根天线的信号独立进行Zak变换。

**物理意义**：
- 延迟轴 → 频率轴
- 时间轴 → 多普勒轴

In [ ]:
def zak_transform(x, N, M):
    """
    离散Zak变换（对称有限傅里叶变换 SFFT）
    
    将延迟-多普勒域信号变换到时频域
    
    参数:
        x: 输入信号，shape (M, N) - 延迟-多普勒域信号
        N: 时间维度
        M: 频率维度
    
    返回:
        X: 时频域信号，shape (N, M)
    
    公式: X[k,l] = (1/N) * sum_{m=0}^{M-1} sum_{n=0}^{N-1} x[m,n] * exp(j2pi*(kn/N + lm/M))
    """
    # 步骤1：沿列方向FFT（多普勒→时间）
    X_temp = np.fft.fft(x, axis=0)
    # 步骤2：沿行方向IFFT（延迟→频率）
    X = np.fft.ifft(X_temp, axis=1)
    return X

def izak_transform(X, N, M):
    """
    逆Zak变换（逆SFFT）
    
    将时频域信号变换到延迟-多普勒域
    
    参数:
        X: 时频域信号，shape (N, M)
        N: 时间维度
        M: 频率维度
    
    返回:
        x: 延迟-多普勒域信号，shape (M, N)
    
    公式: x[m,n] = (1/M) * sum_{l=0}^{M-1} sum_{k=0}^{N-1} X[k,l] * exp(-j2pi*(kn/N + lm/M))
    """
    # 逆变换：步骤1FFT列，步骤2IFFT行
    x_temp = np.fft.fft(X, axis=0)
    x = np.fft.ifft(x_temp, axis=1)
    return x

# 验证：先Zak再IZak应恢复原信号
np.random.seed(42)
test_signal = np.random.randn(M, N) + 1j * np.random.randn(M, N)
recovered = izak_transform(zak_transform(test_signal, N, M), N, M)
error = np.linalg.norm(test_signal - recovered) / np.linalg.norm(test_signal)
print(f"Zak/IZak变换验证 - 恢复误差: {error:.2e}")
print("（误差来自数值精度，应小于1e-15）")

### 4.2 SFFT在MIMO系统中的应用

对于$N_t$根发射天线，每根天线独立进行Zak变换：

$$\mathbf{X}_{TF} = \begin{bmatrix} \mathcal{Z}(X_1) \\ \mathcal{Z}(X_2) \\ \vdots \\ \mathcal{Z}(X_{N_t}) \end{bmatrix}$$

其中$\mathcal{Z}$表示Zak变换（SFFT）。

In [ ]:
def create_qam_constellation(order):
    """创建QAM星座图"""
    if order == 16:
        levels = np.array([-3, -1, 1, 3]) / 3.0
        constellation = []
        for i in levels:
            for j in levels:
                constellation.append(i + 1j * j)
        return np.array(constellation), 4
    elif order == 4:
        return np.array([1+0j, 0+1j, -1+0j, 0-1j]), 2
    else:
        raise ValueError(f"Unsupported QAM order: {order}")

def qam_modulate(bits, constellation):
    """将比特流映射到QAM星座点"""
    bits_per_symbol = int(np.log2(len(constellation)))
    symbols = []
    for i in range(0, len(bits), bits_per_symbol):
        if i + bits_per_symbol <= len(bits):
            symbol_bits = bits[i:i+bits_per_symbol]
            symbol_idx = int(''.join(map(str, symbol_bits)), 2)
            symbols.append(constellation[symbol_idx])
    return np.array(symbols)

def qam_demodulate(symbols, constellation):
    """将QAM符号解映射回比特"""
    bits_per_symbol = int(np.log2(len(constellation)))
    recovered_bits = []
    for symbol in symbols:
        distances = np.abs(constellation - symbol)
        nearest_idx = np.argmin(distances)
        bit_str = format(nearest_idx, f'0{bits_per_symbol}b')
        recovered_bits.extend([int(b) for b in bit_str])
    return np.array(recovered_bits)

# 创建QAM星座
qam_constellation, qam_bits = create_qam_constellation(qam_order)
print(f"QAM阶数: {qam_order}")
print(f"每个符号携带 {qam_bits} 比特")
print(f"星座点数量: {len(qam_constellation)}")

In [ ]:
# 生成MIMO-OTFS发送信号
np.random.seed(42)

# 为每根发射天线生成独立的QAM数据
total_symbols_per_antenna = N * M
total_bits_per_antenna = total_symbols_per_antenna * qam_bits

# 存储每根天线的延迟-多普勒域信号
X_dd_list = []
tx_bits_list = []

for nt in range(N_t):
    # 生成随机比特
    tx_bits = np.random.randint(0, 2, total_bits_per_antenna)
    tx_bits_list.append(tx_bits)
    
    # QAM调制
    qam_symbols = qam_modulate(tx_bits, qam_constellation)
    
    # 放置到延迟-多普勒网格 (M x N)
    X_dd = qam_symbols.reshape(M, N)
    X_dd_list.append(X_dd)

print(f"生成{N_t}根发射天线的延迟-多普勒域信号")
print(f"每天线符号数: {total_symbols_per_antenna}")
print(f"每天线比特数: {total_bits_per_antenna}")
print(f"总比特数: {N_t * total_bits_per_antenna * qam_bits}")

In [ ]:
# 可视化：多天线延迟-多普勒域信号
fig, axes = plt.subplots(1, N_t, figsize=(6*N_t, 5))
if N_t == 1:
    axes = [axes]

for nt in range(N_t):
    im = axes[nt].imshow(np.abs(X_dd_list[nt]), aspect='auto', origin='lower',
                          extent=[0, N, 0, M], cmap='Blues', interpolation='nearest')
    plt.colorbar(im, ax=axes[nt], label='|X|')
    axes[nt].set_xlabel('多普勒索引 (ν)', fontsize=11)
    axes[nt].set_ylabel('延迟索引 (τ)', fontsize=11)
    axes[nt].set_title(f'发射天线#{nt+1}: 延迟-多普勒域信号', fontsize=12)

plt.tight_layout()
plt.show()

print("观察：每根发射天线在延迟-多普勒域携带独立的QAM数据")
print("这些数据通过空间复用同时传输，实现MIMO容量增益")

In [ ]:
# 对每根天线应用Zak变换（SFFT）
X_tf_list = []
for nt in range(N_t):
    X_tf = zak_transform(X_dd_list[nt], N, M)
    X_tf_list.append(X_tf)

print(f"对{N_t}根天线分别进行Zak变换")
print(f"每根天线时频域信号形状: ({N}, {M})")

# 可视化：Zak变换后的时频域信号
fig, axes = plt.subplots(1, N_t, figsize=(6*N_t, 5))
if N_t == 1:
    axes = [axes]

for nt in range(N_t):
    im = axes[nt].imshow(np.abs(X_tf_list[nt]), aspect='auto', origin='lower',
                          extent=[0, M, 0, N], cmap='Greens', interpolation='nearest')
    plt.colorbar(im, ax=axes[nt], label='|X_tf|')
    axes[nt].set_xlabel('频率索引 (f)', fontsize=11)
    axes[nt].set_ylabel('时间索引 (t)', fontsize=11)
    axes[nt].set_title(f'发射天线#{nt+1}: Zak变换后信号', fontsize=12)

plt.tight_layout()
plt.show()

print("观察：Zak变换将稀疏的DD域信号转换为密集的TF域信号")
print("这种扩散性质使得所有QAM符号共享相同的信道响应")

### 4.3 OFDM调制

时频域信号通过OFDM调制转换为时域信号。

In [ ]:
def ofdm_modulate(tf_grid, n_fft, cp_len):
    """
    OFDM调制：将时频域信号转换为时域OFDM符号
    
    参数:
        tf_grid: shape (N_time, M_freq) - 时频域信号
        n_fft: FFT大小
        cp_len: 循环前缀长度
    
    返回:
        ofdm_symbols: shape (N_time, n_fft + cp_len) - 含CP的时域OFDM符号
    """
    num_symbols = tf_grid.shape[0]
    ofdm_symbols = []
    
    for i in range(num_symbols):
        # IFFT：将频域信号转换到时域
        time_domain = np.fft.ifft(tf_grid[i], n=n_fft)
        
        # 添加循环前缀
        cp = time_domain[-cp_len:]
        ofdm_with_cp = np.concatenate([cp, time_domain])
        ofdm_symbols.append(ofdm_with_cp)
    
    return np.array(ofdm_symbols)

# 对每根天线进行OFDM调制
tx_ofdm_list = []
for nt in range(N_t):
    tx_ofdm = ofdm_modulate(X_tf_list[nt], M, cp_len)
    tx_ofdm_list.append(tx_ofdm)

print(f"对{N_t}根天线的时频域信号进行OFDM调制")
print(f"每个OFDM符号长度: {M + cp_len} (数据: {M} + CP: {cp_len})")
print(f"每根天线OFDM符号形状: {tx_ofdm_list[0].shape}")

## 5. MIMO信道建模

### 5.1 延迟-多普勒域MIMO信道

MIMO信道在延迟-多普勒域表示为$N_r \times N_t$的信道矩阵集合：

$$\mathbf{H}(\tau, \nu) = \begin{bmatrix} h_{11}(\tau,\nu) & h_{12}(\tau,\nu) \\ h_{21}(\tau,\nu) & h_{22}(\tau,\nu) \end{bmatrix}$$

每条路径$h_{r,n_t}(\tau, \nu)$在延迟-多普勒域是稀疏的。

In [ ]:
def create_mimo_dd_channel(N_t, N_r, num_paths, max_delay, max_doppler, N, M):
    """
    创建延迟-多普勒域MIMO信道
    
    参数:
        N_t: 发射天线数
        N_r: 接收天线数
        num_paths: 多径数量
        max_delay: 最大延迟
        max_doppler: 最大多普勒
        N: 时间维度
        M: 频率维度
    
    返回:
        H_dd: shape (N_r, N_t, M, N) - 延迟-多普勒域MIMO信道张量
        channel_taps: 路径信息列表
    """
    np.random.seed(100)
    
    # 初始化信道张量: (N_r, N_t, M, N)
    H_dd = np.zeros((N_r, N_t, M, N), dtype=complex)
    
    channel_taps = []
    
    for n_t in range(N_t):
        for n_r in range(N_r):
            # 信道在延迟-多普勒域是稀疏的
            
            # 主径：延迟=0, 多普勒=0
            H_dd[n_r, n_t, 0, max_doppler] = 1.0 + 0j
            channel_taps.append((n_r, n_t, 0, 0, 1.0 + 0j))
            
            # 随机路径
            for p in range(num_paths - 1):
                delay_idx = np.random.randint(1, max_delay + 1)
                doppler_idx = np.random.randint(0, 2 * max_doppler + 1) - max_doppler
                amplitude = (np.random.randn() + 1j * np.random.randn()) * 0.5
                
                if delay_idx < M and (doppler_idx + max_doppler) < N:
                    H_dd[n_r, n_t, delay_idx, doppler_idx + max_doppler] = amplitude
                    channel_taps.append((n_r, n_t, delay_idx, doppler_idx, amplitude))
    
    return H_dd, channel_taps

# 创建MIMO信道
H_dd, channel_taps = create_mimo_dd_channel(N_t, N_r, num_paths, max_delay, max_doppler, N, M)
print(f"MIMO-OTFS信道创建完成")
print(f"信道张量形状: {H_dd.shape} = (N_r={N_r}, N_t={N_t}, M={M}, N={N})")
print(f"路径数量: {len(channel_taps)}")

In [ ]:
# 可视化：MIMO信道在延迟-多普勒域的结构
fig, axes = plt.subplots(N_r, N_t, figsize=(6*N_t, 5*N_r))
if N_r == 1 and N_t == 1:
    axes = [[axes]]
elif N_r == 1:
    axes = [axes]
elif N_t == 1:
    axes = [[a] for a in axes]

for n_t in range(N_t):
    for n_r in range(N_r):
        im = axes[n_r][n_t].imshow(np.abs(H_dd[n_r, n_t]), aspect='auto', origin='lower',
                                    extent=[-max_doppler, max_doppler, 0, max_delay+1],
                                    cmap='hot', interpolation='nearest')
        plt.colorbar(im, ax=axes[n_r][n_t], label='|h|')
        axes[n_r][n_t].set_xlabel('多普勒索引 (ν)', fontsize=11)
        axes[n_r][n_t].set_ylabel('延迟索引 (τ)', fontsize=11)
        axes[n_r][n_t].set_title(f'信道 h_{{{n_r+1},{n_t+1}}}(\\tau, \\nu)', fontsize=12)

plt.tight_layout()
plt.show()

print("观察：每对发射-接收天线之间的信道在延迟-多普勒域是稀疏的")
print("只有几个非零点（几条主要路径）")

In [ ]:
def apply_mimo_channel(tx_ofdm_list, channel_taps, N_t, N_r, max_delay, max_doppler, cp_len, M, N):
    """
    将MIMO信道应用于OTFS信号
    
    参数:
        tx_ofdm_list: list of N_t arrays, each shape (N, M+cp_len)
        channel_taps: 路径信息列表
        N_t, N_r: 天线数
    
    返回:
        rx_ofdm_list: list of N_r arrays, each shape (N, M+cp_len)
    """
    # 初始化接收信号
    rx_ofdm_list = [np.zeros_like(tx_ofdm_list[0], dtype=complex) for _ in range(N_r)]
    
    # 对每条路径进行处理
    for n_r, n_t, delay_idx, doppler_idx, amplitude in channel_taps:
        for sym_idx in range(N):
            # 多普勒相位旋转
            doppler_phase = np.exp(1j * 2 * np.pi * doppler_idx * sym_idx / N)
            gain = amplitude * doppler_phase
            
            # 获取发射天线信号
            tx_signal = tx_ofdm_list[n_t][sym_idx, cp_len:cp_len+M]
            
            # 应用延迟和多普勒
            start = cp_len + delay_idx
            end = start + M
            if end <= rx_ofdm_list[n_r].shape[1]:
                rx_ofdm_list[n_r][sym_idx, start:end] += tx_signal * gain
    
    return rx_ofdm_list

# 通过MIMO信道
rx_ofdm_list = apply_mimo_channel(tx_ofdm_list, channel_taps, N_t, N_r, 
                                   max_delay, max_doppler, cp_len, M, N)
print(f"信号通过{N_t}x{N_r} MIMO信道")
print(f"每根接收天线信号形状: {rx_ofdm_list[0].shape}")

In [ ]:
# 添加噪声
SNR_db = 20
signal_power = sum(np.mean(np.abs(rx)**2) for rx in rx_ofdm_list) / N_r
noise_power = signal_power / (10**(SNR_db/10))

for n_r in range(N_r):
    noise = np.sqrt(noise_power/2) * (np.random.randn(*rx_ofdm_list[n_r].shape) + 
                                       1j * np.random.randn(*rx_ofdm_list[n_r].shape))
    rx_ofdm_list[n_r] += noise

print(f"添加噪声 - SNR={SNR_db}dB")
print(f"噪声功率: {noise_power:.4f}")

## 6. 接收端后处理

### 6.1 OFDM解调 + ISFFT

接收端执行与发送端相反的操作：

1. OFDM解调（去除CP + FFT）
2. ISFFT（逆Zak变换）
3. MIMO检测与均衡

In [ ]:
def ofdm_demodulate(rx_ofdm_symbols, n_fft, cp_len):
    """
    OFDM解调：将时域OFDM符号转换为频域
    
    参数:
        rx_ofdm_symbols: shape (N_time, n_fft + cp_len) - 含CP的接收OFDM符号
        n_fft: FFT大小
        cp_len: 循环前缀长度
    
    返回:
        tf_grid: shape (N_time, n_fft) - 时频域信号
    """
    num_symbols = rx_ofdm_symbols.shape[0]
    tf_grid = []
    
    for i in range(num_symbols):
        # 去除CP
        symbol = rx_ofdm_symbols[i, cp_len:cp_len + n_fft]
        
        # FFT：时域 → 频域
        freq_domain = np.fft.fft(symbol, n=n_fft)
        tf_grid.append(freq_domain)
    
    return np.array(tf_grid)

# 对每根接收天线进行OFDM解调
Y_tf_list = []
for n_r in range(N_r):
    Y_tf = ofdm_demodulate(rx_ofdm_list[n_r], M, cp_len)
    Y_tf_list.append(Y_tf)

print(f"对{N_r}根接收天线进行OFDM解调")
print(f"每根天线时频域信号形状: {Y_tf_list[0].shape}")

In [ ]:
# 对每根接收天线进行ISFFT（逆Zak变换）
Y_dd_list = []
for n_r in range(N_r):
    Y_dd = izak_transform(Y_tf_list[n_r], N, M)
    Y_dd_list.append(Y_dd)

print(f"对{N_r}根接收天线进行ISFFT")
print(f"每根天线延迟-多普勒域信号形状: {Y_dd_list[0].shape}")

In [ ]:
# 可视化：接收信号在延迟-多普勒域
fig, axes = plt.subplots(2, N_r, figsize=(6*N_r, 10))
if N_r == 1:
    axes = [[axes[0], axes[1]]]

# 发射信号（第一根天线）
for n_r in range(N_r):
    im = axes[0][n_r].imshow(np.abs(X_dd_list[0]), aspect='auto', origin='lower',
                              extent=[0, N, 0, M], cmap='Blues', interpolation='nearest')
    plt.colorbar(im, ax=axes[0][n_r], label='|X|')
    axes[0][n_r].set_xlabel('多普勒索引 (ν)', fontsize=11)
    axes[0][n_r].set_ylabel('延迟索引 (τ)', fontsize=11)
    axes[0][n_r].set_title(f'发射天线#1: DD域信号', fontsize=12)

# 接收信号
for n_r in range(N_r):
    im = axes[1][n_r].imshow(np.abs(Y_dd_list[n_r]), aspect='auto', origin='lower',
                              extent=[0, N, 0, M], cmap='Reds', interpolation='nearest')
    plt.colorbar(im, ax=axes[1][n_r], label='|Y|')
    axes[1][n_r].set_xlabel('多普勒索引 (ν)', fontsize=11)
    axes[1][n_r].set_ylabel('延迟索引 (τ)', fontsize=11)
    axes[1][n_r].set_title(f'接收天线#{n_r+1}: DD域信号', fontsize=12)

plt.tight_layout()
plt.show()

print("观察：接收信号是所有发射天线信号的叠加（受信道影响）")
print("需要进行MIMO检测来分离各天线的数据")

## 7. MIMO预编码技术

### 7.1 为什么要预编码？

在MIMO系统中，多个数据流同时传输会相互干扰。预编码技术在发射端对信号进行预处理，利用信道状态信息（CSI）将干扰转化为有益信号或消除干扰。

**MIMO-OTFS预编码的特殊性**：
- 在延迟-多普勒域，信道$\mathbf{H}(\tau, \nu)$是时不变的
- 可以获得准确的全局CSI
- 预编码矩阵设计更简单有效

### 7.2 线性预编码方案

| 方案 | 公式 | 复杂度 | 性能特点 |
|------|------|--------|----------|
| **迫零(ZF)** | $\mathbf{F}_{ZF} = \mathbf{H}^{-1}$ | $O(N_t^3)$ | 完全消除干扰，但噪声放大 |
| **最小均方误差(MMSE)** | $\mathbf{F}_{MMSE} = \mathbf{H}^H(\mathbf{H}\mathbf{H}^H+\sigma^2\mathbf{I})^{-1}$ | $O(N_t^3)$ | 平衡干扰消除与噪声 |
| **最大比传输(MRT)** | $\mathbf{F}_{MRT} = \mathbf{H}^H$ | $O(N_t^2)$ | 最大化信号增益，单流 |
| **脏纸编码(DPC)** | 非线性 | 极高 | 香农容量可达 |

In [ ]:
def zf_precoder(H):
    """
    迫零(ZF)预编码器
    
    F_ZF = H^H * (H * H^H)^(-1) = pinv(H)
    
    参数:
        H: 信道矩阵，shape (N_r, N_t)
    
    返回:
        F: 预编码矩阵，shape (N_t, N_r)
    """
    return np.linalg.pinv(H)

def mmse_precoder(H, SNR_db):
    """
    最小均方误差(MMSE)预编码器
    
    F_MMSE = H^H * (H*H^H + sigma^2*I)^(-1)
    
    参数:
        H: 信道矩阵，shape (N_r, N_t)
        SNR_db: 信噪比(dB)
    
    返回:
        F: 预编码矩阵，shape (N_t, N_r)
    """
    N_r = H.shape[0]
    SNR = 10 ** (SNR_db / 10)
    sigma2 = 1 / SNR
    return H.conj().T @ np.linalg.inv(H @ H.conj().T + sigma2 * np.eye(N_r))

def mrt_precoder(H):
    """
    最大比传输(MRT)预编码器
    
    F_MRT = H^H
    
    参数:
        H: 信道矩阵，shape (N_r, N_t)
    
    返回:
        F: 预编码矩阵，shape (N_t, N_r)
    """
    return H.conj().T

def toep_inverse(H):
    """
    计算H的Toeplitz逆（用于Tomlinson-Harashima预编码）
    这里简化为伪逆
    """
    return np.linalg.pinv(H)

print("预编码器函数定义完成")
print("支持：ZF、MMSE、MRT预编码方案")

In [ ]:
# 演示：不同预编码方案的效果
np.random.seed(42)

# 假设在某个(延迟,多普勒)格点上的信道矩阵
H_example = H_dd[:, :, 0, max_doppler]  # 取主径处的信道
print("示例信道矩阵 H:")
print(H_example)
print(f"\n信道矩阵形状: {H_example.shape} = ({N_r}, {N_t})")
print(f"信道条件数: {np.linalg.cond(H_example):.2f}")

In [ ]:
# 计算不同预编码矩阵
F_zf = zf_precoder(H_example)
F_mmse = mmse_precoder(H_example, SNR_db)
F_mrt = mrt_precoder(H_example)

print("ZF预编码矩阵 F_ZF:")
print(F_zf)

print("\nMMSE预编码矩阵 F_MMSE (SNR=20dB):")
print(F_mmse)

print("\nMRT预编码矩阵 F_MRT:")
print(F_mrt)

In [ ]:
# 验证预编码效果：发射两个独立数据流
s = np.array([1+1j, -1-1j])  # 两个独立符号

# 无预编码直接发射
y_no_precoding = H_example @ s

# ZF预编码
x_zf = F_zf @ s
y_zf = H_example @ x_zf

# MMSE预编码
x_mmse = F_mmse @ s
y_mmse = H_example @ x_mmse

print("发射符号 s:", s)
print("\n无预编码 - 接收:", y_no_precoding, "误差:", np.linalg.norm(y_no_precoding - s))
print("ZF预编码 - 接收:", y_zf, "误差:", np.linalg.norm(y_zf - s))
print("MMSE预编码 - 接收:", y_mmse, "误差:", np.linalg.norm(y_mmse - s))

print("\n观察：ZF预编码完全消除了信道引起的干扰")
print("MMSE预编码在干扰消除和噪声放大之间取得平衡")

### 7.3 Tomlinson-Harashima预编码

TH预编码是一种非线性预编码技术，通过模运算和脏纸编码思想实现：

**原理**：
1. 发射机已知信道矩阵$\mathbf{H}$
2. 按顺序消除各数据流的干扰
3. 使用模运算限制发射功率

**公式**：
对于第$i$个数据流：

$$x_i = s_i - \sum_{j=1}^{i-1} h_{ij} x_j \quad \text{mod} \quad \mathcal{C}$$

其中$\mathcal{C}$是QAM星座的复数网格。

In [ ]:
def th_precoder(H, s, constellation):
    """
    Tomlinson-Harashima预编码器（简化版）
    
    顺序消除各数据流的干扰
    
    参数:
        H: 信道矩阵，shape (N_r, N_t)
        s: 发送符号向量，length N_t
        constellation: QAM星座点数组
    
    返回:
        x: 预编码后的发射符号
    """
    N_t = len(s)
    x = np.zeros(N_t, dtype=complex)
    
    # 获取星座的网格间距
    constellation_real = np.concatenate([np.real(constellation), np.imag(constellation)])
    spacing = np.min(np.diff(np.unique(constellation_real)))
    mod_region = 2  # 模运算区域大小（相对于星座间距）
    
    for i in range(N_t):
        # 计算来自已发射符号的干扰
        interference = 0
        for j in range(i):
            interference += H[i, j] * x[j]
        
        # 消除干扰（使用信道逆的近似）
        if i == 0:
            x[i] = s[i] / H[i, i]
        else:
            # 使用逐次干扰消除
            h_row = H[i, :i+1]
            x[:i+1] = np.linalg.solve(h_row.reshape(1, -1) @ np.eye(i+1), 
                                      np.append(s[:i], s[i] - interference).reshape(-1, 1)).flatten()
        
        # 模运算以限制功率
        real_part = np.real(x[i])
        imag_part = np.imag(x[i])
        real_part = real_part - mod_region * spacing * np.round(real_part / (mod_region * spacing))
        imag_part = imag_part - mod_region * spacing * np.round(imag_part / (mod_region * spacing))
        x[i] = real_part + 1j * imag_part
    
    return x

# 演示TH预编码
s_example = np.array([1+1j, -1-1j, 1-1j, -1+1j])[:N_t]
x_th = th_precoder(H_example[:N_t, :], s_example, qam_constellation)
y_th = H_example[:N_t, :] @ x_th

print("Tomlinson-Harashima预编码演示")
print(f"发射符号 s: {s_example}")
print(f"TH预编码后 x: {x_th}")
print(f"接收符号 y: {y_th}")
print(f"发射功率对比 - 原始: {np.mean(np.abs(s_example)**2):.2f}, TH后: {np.mean(np.abs(x_th)**2):.2f}")

## 8. 联合检测算法

### 8.1 检测问题的数学表述

在MIMO-OTFS接收端，需要从接收信号$\mathbf{Y}$中恢复发射信号$\mathbf{X}$。

**系统方程**：

$$\mathbf{Y} = \mathbf{H} \mathbf{X} + \mathbf{N}$$

其中：
- $\mathbf{Y} \in \mathbb{C}^{N_r \times MN}$：接收信号
- $\mathbf{H} \in \mathbb{C}^{N_r \times N_t}$：MIMO信道矩阵
- $\mathbf{X} \in \mathbb{C}^{N_t \times MN}$：发射信号
- $\mathbf{N} \in \mathbb{C}^{N_r \times MN}$：噪声

### 8.2 最大似然检测（ML）

ML检测寻找使似然函数最大的发射信号：

$$\hat{\mathbf{X}}_{ML} = \arg\max_{\mathbf{X} \in \mathcal{C}^{N_t}} p(\mathbf{Y}|\mathbf{X}, \mathbf{H})$$

**优点**：最优性能
**缺点**：复杂度$O(|\mathcal{C}|^{N_t})$，随天线数和调制阶数指数增长

对于$N_t=2, 16\text{-QAM}$：$16^2 = 256$次搜索

In [ ]:
def ml_detector(H, y, constellation):
    """
    最大似然(ML)检测器
    
    穷举所有可能的发射符号组合
    
    参数:
        H: 信道矩阵，shape (N_r, N_t)
        y: 接收符号，shape (N_r,)
        constellation: QAM星座点数组
    
    返回:
        x_hat: 检测到的符号
    """
    N_t = H.shape[1]
    constellation = constellation.flatten()
    
    min_distance = float('inf')
    best_x = None
    
    # 生成所有可能的发射符号组合
    from itertools import product
    for x_combination in product(constellation, repeat=N_t):
        x = np.array(x_combination)
        # 计算欧氏距离
        distance = np.linalg.norm(y - H @ x)
        if distance < min_distance:
            min_distance = distance
            best_x = x
    
    return best_x

def mf_detector(H, y):
    """
    匹配滤波器(MF)检测器
    
    简单的线性检测，不考虑干扰
    
    参数:
        H: 信道矩阵
        y: 接收符号
    
    返回:
        x_hat: 检测到的符号
    """
    return H.conj().T @ y

def zf_detector(H, y):
    """
    迫零(ZF)检测器
    
    完全消除干扰，但放大噪声
    
    参数:
        H: 信道矩阵
        y: 接收符号
    
    返回:
        x_hat: 检测到的符号
    """
    return np.linalg.pinv(H) @ y

def mmse_detector(H, y, SNR_db):
    """
    最小均方误差(MMSE)检测器
    
    平衡干扰消除与噪声放大
    
    参数:
        H: 信道矩阵
        y: 接收符号
        SNR_db: 信噪比(dB)
    
    返回:
        x_hat: 检测到的符号
    """
    N_r = H.shape[0]
    SNR = 10 ** (SNR_db / 10)
    sigma2 = 1 / SNR
    return H.conj().T @ np.linalg.inv(H @ H.conj().T + sigma2 * np.eye(N_r)) @ y

print("检测器函数定义完成")
print("支持：ML、MF、ZF、MMSE检测器")

### 8.3 MAP检测与消息传递

MAP（最大后验概率）检测最大化后验概率：

$$\hat{\mathbf{X}}_{MAP} = \arg\max_{\mathbf{X}} p(\mathbf{X}|\mathbf{Y}) = \arg\max_{\mathbf{X}} p(\mathbf{Y}|\mathbf{X}) p(\mathbf{X})$$

**消息传递检测器（MP）**利用因子图结构，在变量节点和函数节点之间迭代传递消息。

**优势**：
- 复杂度低于ML：$O(N_t^2)$ per iteration
- 可利用信道稀疏性
- 性能接近ML

In [ ]:
def map_detector(H, y, constellation, prior=None):
    """
    MAP（最大后验概率）检测器
    
    考虑先验概率的检测
    
    参数:
        H: 信道矩阵
        y: 接收符号
        constellation: QAM星座
        prior: 先验概率分布（可选）
    
    返回:
        x_hat: 检测到的符号
    """
    N_t = H.shape[1]
    constellation = constellation.flatten()
    
    if prior is None:
        prior = np.ones(len(constellation)) / len(constellation)
    
    best_posterior = -np.inf
    best_x = None
    
    from itertools import product
    for x_combination in product(constellation, repeat=N_t):
        x = np.array(x_combination)
        
        # 计算似然
        noise_var = 0.01  # 假设已知噪声方差
        likelihood = np.exp(-np.linalg.norm(y - H @ x)**2 / (2 * noise_var))
        
        # 计算后验
        posterior = likelihood * np.prod([prior[np.where(constellation == xi)[0][0]] for xi in x])
        
        if posterior > best_posterior:
            best_posterior = posterior
            best_x = x
    
    return best_x


def message_passing_detector(H, y, constellation, max_iter=10):
    """
    消息传递检测器（简化版）
    
    在MIMO信道上迭代传递消息
    
    参数:
        H: 信道矩阵，shape (N_r, N_t)
        y: 接收符号
        constellation: QAM星座
        max_iter: 最大迭代次数
    
    返回:
        x_hat: 检测到的符号
    """
    N_t = H.shape[1]
    N_r = H.shape[0]
    constellation = constellation.flatten()
    
    # 初始化：每个变量节点的概率分布
    probs = [np.ones(len(constellation)) / len(constellation) for _ in range(N_t)]
    
    for iteration in range(max_iter):
        # 简化的消息传递：重新计算后验概率
        for i in range(N_t):
            new_probs = np.zeros(len(constellation))
            for c_idx, c in enumerate(constellation):
                # 计算边际似然
                x_hypothesis = np.zeros(N_t, dtype=complex)
                probs_product = 1
                for j in range(N_t):
                    if j != i:
                        # 使用当前概率估计
                        probs_product *= probs[j][np.argmax(probs[j])]
                
                x_hypothesis[i] = c
                # 其他符号用均值估计
                for j in range(N_t):
                    if j != i:
                        x_hypothesis[j] = constellation[np.argmax(probs[j])]
                
                noise_var = 0.01
                likelihood = np.exp(-np.linalg.norm(y - H @ x_hypothesis)**2 / (2 * noise_var))
                new_probs[c_idx] = likelihood * probs_product
            
            # 归一化
            probs[i] = new_probs / np.sum(new_probs)
    
    # 选择概率最大的符号
    x_hat = np.array([constellation[np.argmax(probs[i])] for i in range(N_t)])
    
    return x_hat

print("MAP和消息传递检测器定义完成")

In [ ]:
# 演示不同检测器的性能
np.random.seed(42)

# 信道和信号设置
H_demo = H_example
s_demo = np.array([1+1j, -1-1j])  # 原始符号

# 添加噪声
SNR_demo = 20
noise_var = 1 / (10 ** (SNR_demo / 10))
n_demo = np.sqrt(noise_var/2) * (np.random.randn(N_r) + 1j * np.random.randn(N_r))
y_demo = H_demo @ s_demo + n_demo

print(f"发射符号: {s_demo}")
print(f"接收符号（含噪声）: {y_demo}")
print()

# MF检测
s_mf = mf_detector(H_demo, y_demo)
print(f"MF检测结果: {s_mf}")

# ZF检测
s_zf = zf_detector(H_demo, y_demo)
print(f"ZF检测结果: {s_zf}")

# MMSE检测
s_mmse = mmse_detector(H_demo, y_demo, SNR_demo)
print(f"MMSE检测结果: {s_mmse}")

# ML检测
s_ml = ml_detector(H_demo, y_demo, qam_constellation)
print(f"ML检测结果: {s_ml}")

print("\n检测误差（相对于发射符号）：")
print(f"MF: {np.linalg.norm(s_mf - s_demo):.4f}")
print(f"ZF: {np.linalg.norm(s_zf - s_demo):.4f}")
print(f"MMSE: {np.linalg.norm(s_mmse - s_demo):.4f}")
print(f"ML: {np.linalg.norm(s_ml - s_demo):.4f} (最优)")

## 9. 完整MIMO-OTFS系统演示

### 9.1 系统流程

```
发送端：
QAM符号 → DD域放置 → Zak变换(SFFT) → TF域 → OFDM调制 → 时域
                                              ↓
                                          MIMO预编码

信道：
延迟-多普勒域稀疏MIMO信道 + 高斯噪声

接收端：
时域 → OFDM解调 → TF域 → ISFFT(IZak) → DD域 → MIMO检测 → QAM解映射
```

In [ ]:
def mimo_otfs_system(N_t, N_r, N, M, qam_constellation, SNR_db, 
                     use_precoding=False, precoder_type='zf',
                     use_detection=True, detector_type='mmse'):
    """
    完整MIMO-OTFS系统仿真
    
    参数:
        N_t, N_r: 发射/接收天线数
        N, M: OTFS网格维度
        qam_constellation: QAM星座
        SNR_db: 信噪比(dB)
        use_precoding: 是否使用预编码
        precoder_type: 'zf', 'mmse', 'mrt'
        use_detection: 是否使用MIMO检测
        detector_type: 'mf', 'zf', 'mmse', 'ml'
    
    返回:
        ber: 误码率
        tx_bits: 发射比特
        rx_bits: 接收比特
    """
    np.random.seed(np.random.randint(0, 10000))
    
    # ========== 发送端 ==========
    # 生成QAM数据
    bits_per_symbol = int(np.log2(len(qam_constellation)))
    total_symbols = N_t * N * M
    total_bits = total_symbols * bits_per_symbol
    
    tx_bits = np.random.randint(0, 2, total_bits)
    tx_symbols_all = qam_modulate(tx_bits, qam_constellation)
    
    # 分割为N_t个流
    symbols_per_stream = N * M
    tx_streams_dd = []
    for nt in range(N_t):
        start = nt * symbols_per_stream
        end = start + symbols_per_stream
        X_dd = tx_symbols_all[start:end].reshape(M, N)
        tx_streams_dd.append(X_dd)
    
    # Zak变换
    tx_streams_tf = []
    for nt in range(N_t):
        X_tf = zak_transform(tx_streams_dd[nt], N, M)
        tx_streams_tf.append(X_tf)
    
    # OFDM调制
    tx_ofdm = []
    for nt in range(N_t):
        tx_ofdm.append(ofdm_modulate(tx_streams_tf[nt], M, cp_len))
    
    # ========== 信道 ==========
    # 创建MIMO信道
    H_dd, channel_taps = create_mimo_dd_channel(N_t, N_r, num_paths, 
                                                max_delay, max_doppler, N, M)
    
    # 应用信道（这里简化为在主径格点处理）
    H_main = H_dd[:, :, 0, max_doppler]  # 取主径信道
    
    # 预编码（如果启用）
    if use_precoding:
        if precoder_type == 'zf':
            F = zf_precoder(H_main)
        elif precoder_type == 'mmse':
            F = mmse_precoder(H_main, SNR_db)
        else:  # mrt
            F = mrt_precoder(H_main)
        
        # 对每根天线的信号应用预编码
        for nt in range(N_t):
            tx_ofdm[nt] = tx_ofdm[nt] * np.linalg.norm(H_main) / np.linalg.norm(F[:, nt])
    
    # 通过信道
    rx_ofdm = apply_mimo_channel(tx_ofdm, channel_taps, N_t, N_r,
                                max_delay, max_doppler, cp_len, M, N)
    
    # 添加噪声
    signal_power = sum(np.mean(np.abs(rx)**2) for rx in rx_ofdm) / N_r
    noise_power = signal_power / (10**(SNR_db/10))
    for nt in range(N_r):
        noise = np.sqrt(noise_power/2) * (np.random.randn(*rx_ofdm[nt].shape) + 
                                           1j * np.random.randn(*rx_ofdm[nt].shape))
        rx_ofdm[nt] += noise
    
    # ========== 接收端 ==========
    # OFDM解调
    rx_tf = []
    for nr in range(N_r):
        rx_tf.append(ofdm_demodulate(rx_ofdm[nr], M, cp_len))
    
    # ISFFT
    rx_dd = []
    for nr in range(N_r):
        rx_dd.append(izak_transform(rx_tf[nr], N, M))
    
    # MIMO检测
    # 在主径格点进行检测
    rx_main = np.array([rx_dd[nr][0, max_doppler] for nr in range(N_r)])
    
    if use_detection:
        if detector_type == 'mf':
            tx_recovered = mf_detector(H_main, rx_main)
        elif detector_type == 'zf':
            tx_recovered = zf_detector(H_main, rx_main)
        elif detector_type == 'mmse':
            tx_recovered = mmse_detector(H_main, rx_main, SNR_db)
        elif detector_type == 'ml':
            tx_recovered = ml_detector(H_main, rx_main, qam_constellation)
        else:
            tx_recovered = mmse_detector(H_main, rx_main, SNR_db)
    else:
        tx_recovered = rx_main
    
    # QAM解映射
    rx_symbols = tx_recovered.flatten()
    rx_bits = qam_demodulate(rx_symbols, qam_constellation)
    
    # 计算BER
    n_errors = np.sum(tx_bits[:len(rx_bits)] != rx_bits)
    ber = n_errors / len(rx_bits)
    
    return ber, tx_bits, rx_bits

print("完整MIMO-OTFS系统函数定义完成")

In [ ]:
# 运行完整系统仿真
print("=" * 60)
print("MIMO-OTFS系统仿真")
print("=" * 60)
print(f"系统配置: {N_t}x{N_r} MIMO, {N}x{M} OTFS网格")
print(f"调制方式: {qam_order}-QAM")
print()

# 测试不同配置
configs = [
    ('无检测', False, 'N/A'),
    ('MF检测', True, 'mf'),
    ('ZF检测', True, 'zf'),
    ('MMSE检测', True, 'mmse'),
    ('ML检测', True, 'ml'),
]

SNR_test = 20

results = []
for name, use_det, det_type in configs:
    ber, tx_bits, rx_bits = mimo_otfs_system(
        N_t, N_r, N, M, qam_constellation, SNR_test,
        use_precoding=False,
        use_detection=use_det,
        detector_type=det_type
    )
    results.append((name, ber))
    print(f"{name}: BER = {ber:.6f}")

print()
print("观察：")
print("  - 无检测时BER很高（接近0.5）")
print("  - MF检测效果较差")
print("  - ZF检测消除干扰但放大噪声")
print("  - MMSE在干扰和噪声间取得平衡")
print("  - ML检测最优但复杂度最高")

In [ ]:
# 可视化BER性能对比
fig, ax = plt.subplots(figsize=(10, 6))

names = [r[0] for r in results]
bers = [r[1] for r in results]

bars = ax.bar(names, bers, color=['gray', 'blue', 'green', 'orange', 'red'])
ax.set_ylabel('误码率 (BER)', fontsize=12)
ax.set_title(f'MIMO-OTFS检测器性能对比 ({N_t}x{N_r}, SNR={SNR_test}dB)', fontsize=14)
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')

for bar, ber in zip(bars, bers):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{ber:.2e}',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# 测试预编码的影响
print("=" * 60)
print("预编码效果对比")
print("=" * 60)

precoding_results = []

for snr in [10, 15, 20, 25, 30]:
    # 无预编码 + MMSE检测
    ber_no_prec, _, _ = mimo_otfs_system(
        N_t, N_r, N, M, qam_constellation, snr,
        use_precoding=False, use_detection=True, detector_type='mmse'
    )
    
    # ZF预编码 + MMSE检测
    ber_zf_prec, _, _ = mimo_otfs_system(
        N_t, N_r, N, M, qam_constellation, snr,
        use_precoding=True, precoder_type='zf',
        use_detection=True, detector_type='mmse'
    )
    
    # MMSE预编码 + MMSE检测
    ber_mmse_prec, _, _ = mimo_otfs_system(
        N_t, N_r, N, M, qam_constellation, snr,
        use_precoding=True, precoder_type='mmse',
        use_detection=True, detector_type='mmse'
    )
    
    precoding_results.append((snr, ber_no_prec, ber_zf_prec, ber_mmse_prec))
    print(f"SNR={snr}dB: 无预编码={ber_no_prec:.4e}, ZF预编码={ber_zf_prec:.4e}, MMSE预编码={ber_mmse_prec:.4e}")

In [ ]:
# 可视化预编码效果
fig, ax = plt.subplots(figsize=(10, 6))

snrs = [r[0] for r in precoding_results]
ber_no_prec = [r[1] for r in precoding_results]
ber_zf = [r[2] for r in precoding_results]
ber_mmse = [r[3] for r in precoding_results]

ax.semilogy(snrs, ber_no_prec, 'b-o', linewidth=2, markersize=8, label='无预编码')
ax.semilogy(snrs, ber_zf, 'g-s', linewidth=2, markersize=8, label='ZF预编码')
ax.semilogy(snrs, ber_mmse, 'r-^', linewidth=2, markersize=8, label='MMSE预编码')

ax.set_xlabel('SNR (dB)', fontsize=12)
ax.set_ylabel('误码率 (BER)', fontsize=12)
ax.set_title('MIMO-OTFS预编码性能对比', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(10, 30)

plt.tight_layout()
plt.show()

print("观察：")
print("  - 预编码在高SNR区域效果更明显")
print("  - MMSE预编码整体优于ZF预编码")
print("  - 预编码+检测的联合优化可以获得最佳性能")

## 10. MIMO-OTFS信道容量分析

### 10.1 MIMO信道容量

对于给定信道矩阵$\mathbf{H}$，MIMO信道容量为：

$$C = \log_2 \det\left(\mathbf{I}_{N_r} + \frac{\rho}{N_t} \mathbf{H}\mathbf{H}^H\right)$$

其中$\rho$是接收端SNR。

### 10.2 OTFS对MIMO容量的影响

OTFS在延迟-多普勒域处理信道带来以下优势：

| 优势 | 说明 |
|------|------|
| **更好的条件数** | 所有时频资源经历相同的等价信道 |
| **多普勒分集** | 高移动性场景下仍保持容量 |
| **简化均衡** | 单抽头均衡减少错误传播 |

In [ ]:
def mimo_channel_capacity(H, SNR_db):
    """
    计算MIMO信道容量
    
    C = log2(det(I + rho/N_t * H*H^H))
    """
    N_t = H.shape[1]
    N_r = H.shape[0]
    SNR = 10 ** (SNR_db / 10)
    
    H_H = H.conj().T
    matrix = np.eye(N_r) + (SNR / N_t) * H @ H_H
    
    eigenvalues = np.linalg.eigvalsh(matrix)
    eigenvalues = np.maximum(eigenvalues, 1e-10)
    
    capacity = np.sum(np.log2(eigenvalues))
    return capacity

# 分析MIMO-OTFS信道特性
print("MIMO-OTFS信道分析")
print("=" * 60)

# 分析不同延迟-多普勒格点的信道
snr_db = 20

capacities = []
condition_numbers = []

for delay_idx in range(min(max_delay + 1, M)):
    for doppler_idx in range(N):
        H_slice = H_dd[:, :, delay_idx, doppler_idx]
        
        # 检查是否为零矩阵
        if np.linalg.norm(H_slice) > 1e-6:
            cap = mimo_channel_capacity(H_slice, snr_db)
            cond = np.linalg.cond(H_slice)
            capacities.append(cap)
            condition_numbers.append(cond)

print(f"分析的格点数: {len(capacities)}")
print(f"平均容量: {np.mean(capacities):.2f} bits/s/Hz")
print(f"容量范围: {np.min(capacities):.2f} - {np.max(capacities):.2f}")
print(f"平均条件数: {np.mean(condition_numbers):.2f}")
print(f"条件数范围: {np.min(condition_numbers):.2f} - {np.max(condition_numbers):.2f}")

In [ ]:
# 可视化信道容量分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 容量直方图
ax1 = axes[0]
ax1.hist(capacities, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
ax1.axvline(x=np.mean(capacities), color='red', linestyle='--', linewidth=2, 
            label=f'均值: {np.mean(capacities):.2f}')
ax1.set_xlabel('信道容量 (bits/s/Hz)', fontsize=11)
ax1.set_ylabel('频数', fontsize=11)
ax1.set_title('MIMO-OTFS信道容量分布', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 条件数直方图
ax2 = axes[1]
ax2.hist(condition_numbers, bins=20, color='coral', edgecolor='black', alpha=0.7)
ax2.axvline(x=np.mean(condition_numbers), color='red', linestyle='--', linewidth=2,
            label=f'均值: {np.mean(condition_numbers):.2f}')
ax2.set_xlabel('条件数', fontsize=11)
ax2.set_ylabel('频数', fontsize=11)
ax2.set_title('MIMO信道条件数分布', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("观察：")
print("  - 信道容量在不同格点间变化较小（OTFS的优势）")
print("  - 条件数分布影响空间复用效率")
print("  - 低条件数意味着更好的空间复用潜力")

## 11. 思考题

### 思考题 1：MIMO-OTFS系统模型

解释为什么在MIMO-OTFS系统中，延迟-多普勒域的信道矩阵$\mathbf{H}(\tau, \nu)$对所有时频资源是相同的（时不变）。这种特性如何简化MIMO预编码和检测的设计？

### 思考题 2：预编码技术比较

对比ZF、MMSE和Tomlinson-Harashima预编码：
- 在高SNR区域，哪种预编码方案性能最好？为什么？
- 在低SNR区域呢？
- TH预编码相比线性预编码有什么优势和劣势？

### 思考题 3：联合检测算法

假设一个$4\times 4$ MIMO-OTFS系统，使用16-QAM调制：
- ML检测需要多少次距离计算？
- 如果使用MMSE检测，复杂度如何？
- 消息传递检测器如何利用信道稀疏性来降低复杂度？

### 思考题 4：OTFS与MIMO-OFDM对比

在高速移动场景（车速350 km/h，载波频率2.6 GHz）下：
- 估计最大多普勒频移
- MIMO-OFDM会遇到什么问题（ICI、导频污染等）？
- MIMO-OTFS如何利用延迟-多普勒域表示来克服这些问题？

### 思考题 5：容量分析

理论分析表明，在高SNR区域，MIMO容量随$\min(N_t, N_r)$线性增长。但在实际系统中，以下因素会限制这一增长：
1. 信道条件数（空间相关性）
2. 导频污染
3. 反馈延迟

请分析每个因素如何影响MIMO-OTFS系统的实际容量。

### 思考题 6：实际系统考虑

设计一个实际的车联网（V2X）MIMO-OTFS系统：
- 系统参数选择（$N_t, N_r, N, M$）
- 预编码和检测方案选择
- 信道估计方案
- 如何处理多普勒扩展

---

## 总结

本notebook介绍了MIMO-OTFS系统的完整技术框架：

### 核心概念

1. **MIMO-OTFS系统模型**：
   - 在延迟-多普勒域进行MIMO信号处理
   - 信道$\mathbf{H}(\tau, \nu)$对所有时频资源时不变
   - 结合MIMO空间复用与OTFS时不变信道优势

2. **发送端预处理**：
   - Zak变换（SFFT）连接延迟-多普勒域与时频域
   - OFDM调制将时频域信号转换为时域
   - 预编码技术在发射端消除干扰

3. **接收端后处理**：
   - OFDM解调
   - 逆Zak变换（ISFFT/IZak）
   - MIMO检测器恢复发送符号

4. **MIMO预编码技术**：
   - ZF：完全消除干扰，但放大噪声
   - MMSE：最优干扰-噪声平衡
   - TH：非线性预编码，接近DPC性能

5. **联合检测算法**：
   - ML：最优但复杂度指数级
   - ZF/MMSE：线性复杂度
   - 消息传递：复杂度与迭代次数线性相关

### 与前面notebook的关联

- **OTFS调制**：延迟-多普勒域处理、SFFT/ISFFT变换
- **MIMO基础**：空间复用、预编码、均衡
- **Zak变换**：连接DD域与TF域的数学工具

MIMO-OTFS代表了未来高移动性无线通信系统的重要发展方向。
